In [19]:
import pandas as pd

In [20]:
sentences = [
    "i love deep learning",
    "i love transformers",
    "attention is powerful",
    "deep learning uses attention",
    "transformers are neural networks"
]

In [21]:
def processAndTokenize(sentence:str):
    return sentence.strip().lower().split()

In [22]:
for i in range(len(sentences)):
    sentences[i] = processAndTokenize(sentences[i])


In [23]:
sentences

[['i', 'love', 'deep', 'learning'],
 ['i', 'love', 'transformers'],
 ['attention', 'is', 'powerful'],
 ['deep', 'learning', 'uses', 'attention'],
 ['transformers', 'are', 'neural', 'networks']]

In [24]:
from collections import Counter

In [25]:
vocab = Counter()
for tokens in sentences:
    vocab.update(tokens)

In [26]:
vocab

Counter({'i': 2,
         'love': 2,
         'deep': 2,
         'learning': 2,
         'transformers': 2,
         'attention': 2,
         'is': 1,
         'powerful': 1,
         'uses': 1,
         'are': 1,
         'neural': 1,
         'networks': 1})

In [27]:
wordsCount = len(vocab)
wordIndex = {}
index = 1

In [28]:
for word, _ in vocab.items():
    if word not in wordIndex:
        wordIndex[word] = index
        index+=1

In [29]:
indexedSentences = []
for tokens in sentences:
    indexedVector = []
    for token in tokens:
        indexedVector.append(wordIndex[token])
    indexedSentences.append(indexedVector)

In [30]:
indexedSentences

[[1, 2, 3, 4], [1, 2, 5], [6, 7, 8], [3, 4, 9, 6], [5, 10, 11, 12]]

In [31]:
import random
embeedingSize = 8
embeedingMatrix = [[random.uniform(-0.5,0.5) for _ in range(embeedingSize)] for _ in range(wordsCount)]

In [ ]:
attentionDimensions = 5
heads = 2
Wks = []
Wqs = []
Wvs = []
for i in range(heads):
    Wk = [[random.uniform(-0.5, 0.5) for _ in range(attentionDimensions)] for _ in range(embeedingSize)]
    Wq = [[random.uniform(-0.5, 0.5) for _ in range(attentionDimensions)] for _ in range(embeedingSize)]
    Wv = [[random.uniform(-0.5, 0.5) for _ in range(attentionDimensions)] for _ in range(embeedingSize)]
    Wks.append(Wk)
    Wqs.append(Wqs)
    Wvs.append(Wv)

In [33]:
import math

In [ ]:
Wy = [[random.uniform(-0.5, 0.5) for _ in range(wordsCount)] for _ in range(embeedingSize)]
Wo = [[random.random(-0.5, 0.5) for _ in range(embeedingSize)] for _ in range(attentionDimensions*heads)]

In [ ]:
hlayer1Dimensions = 32
Wh1 = [[random.uniform(-0.5, 0.5) for _ in range(hlayer1Dimensions)] for _ in range(embeedingSize)]

In [ ]:
Wh2 = [[random.uniform(-0.5, 0.5) for _ in range(embeedingSize)] for _ in range(hlayer1Dimensions)]

In [ ]:
maxSequenceLength = 20
positionalEncoding = [[0.0 for _ in range(embeedingSize)] for _ in range(maxSequenceLength)]

In [ ]:
for pos in range(maxSequenceLength):
    for i in range(embeedingSize):
        denominator = (
            10000
            **
            ((2 * (i // 2)) / embeedingSize)
        )

        if i % 2 == 0:

            positionalEncoding[pos][i] = math.sin(
                pos / denominator
            )

        else:

            positionalEncoding[pos][i] = math.cos(
                pos / denominator
            )       

In [ ]:
lr = 0.001

In [ ]:
for tokens in indexedSentences:
    headValues = []
    x_embeedings = []
    for head in range(heads):
        sequenceSize = len(tokens)

        Q = [[0.0 for _ in range(attentionDimensions)] 
            for _ in range(sequenceSize)]

        K = [[0.0 for _ in range(attentionDimensions)] 
            for _ in range(sequenceSize)]
        
        V = [[0.0 for _ in range(attentionDimensions)] 
            for _ in range(sequenceSize)]

        # compute Q and K
        for ind, token in enumerate(tokens):

            x = embeedingMatrix[token-1]
            for i in range(embeedingSize):
                x[i]+=positionalEncoding[ind][i]
            x_embeedings.append(x)

            for outDim in range(attentionDimensions):

                for inDim in range(embeedingSize):

                    Q[ind][outDim] += (
                        x[inDim]
                        *
                        Wq[head][inDim][outDim]
                    )

                    K[ind][outDim] += (
                        x[inDim]
                        *
                        Wk[head][inDim][outDim]
                    )

                    V[ind][outDim] += (
                        x[inDim]
                        *
                        Wv[head][inDim][outDim]
                    )



        # compute attention scores
        QK = [[0.0 for _ in range(sequenceSize)] 
            for _ in range(sequenceSize)]

        for i in range(sequenceSize):

            for j in range(sequenceSize):

                for k in range(attentionDimensions):

                    QK[i][j] += (
                        Q[i][k]
                        *
                        K[j][k]
                    )
                QK[i][j]/=math.sqrt(attentionDimensions)

        attentionWeights = [[0.0 for _ in range(sequenceSize)] for _ in range(sequenceSize)]

        #masking future words
        for i in range(sequenceSize):
            for j in range(i+1,sequenceSize):
                QK[i][j] = -math.inf
        
        for i in range(sequenceSize):
            denominator = 0
            for j in range(sequenceSize):
                denominator+=math.exp(QK[j][i])
            for j in range(sequenceSize):
                attentionWeights[j][i] = math.exp(QK[j][i])/denominator

        values = [[0.0 for _ in range(attentionDimensions)] for _ in range(sequenceSize)]
        
        for i in range(sequenceSize):
            for j in range(sequenceSize):
                for k in range(attentionDimensions):
                    values[i][k]+=attentionWeights[i][j]*V[j][k]
        headValues.append(values)


    concatinatedValue = []
    for i in range(sequenceSize):
        concatinatedVertex = []
        for j in range(heads):
            concatinatedVertex.extend(headValues[j][i])
        concatinatedValue.append(concatinatedVertex)

    projected = [[0.0 for _ in range(embeedingSize)] for _ in range(sequenceSize)]

    for i in range(sequenceSize):
        for j in range(embeedingSize):
            for k in range(heads*attentionDimensions):
                projected[i][j]+=concatinatedValue[i][k]*Wo[k][j]

    #RESNET (Residual Connection)
    residualOutput = [[0.0 for _ in range(embeedingSize)] for _ in range(sequenceSize)]
    for i in range(sequenceSize):
        for j in range(embeedingSize):
            residualOutput[i][j] = x_embeedings[i][j] + projected[i][j]
    
    #layer normalization
    layerNormOutput = [[0.0 for _ in range(embeedingSize)] for _ in range(sequenceSize)]
    epsilon = 1e-5
    for i in range(sequenceSize):
        mean = sum(residualOutput[i])/embeedingSize
        variance = sum([(x-mean)**2 for x in residualOutput[i]])/embeedingSize
        for j in range(embeedingSize):
            layerNormOutput[i][j] = (residualOutput[i][j]-mean)/math.sqrt(variance+epsilon)

    #feed forward neural network
    #layer 1 we are enlarging dimensions to hidden layer 1

    h1Output = [[0.0 for _ in range(hlayer1Dimensions)] for _ in range(sequenceSize)]
    for i in range(sequenceSize):
        for j in range(hlayer1Dimensions):
            for k in range(embeedingSize):
                h1Output[i][j]+=(layerNormOutput[i][k]*Wh1[k][j])

    #Apply relu
    for i in range(sequenceSize):
        for j in range(hlayer1Dimensions):
            h1Output[i][j] = max(0, h1Output[i][j])

    #hidden layer 2 for output(squishes)
    h2Output = [[0.0 for _ in range(embeedingSize)] for _ in range(sequenceSize)]

    for i in range(sequenceSize):
        for j in range(embeedingSize):
            for k in range(hlayer1Dimensions):
                h2Output[i][j]+=(h1Output[i][k]*Wh2[k][j])

    #Applying resnet
    fnnResnetOutput = [[0.0 for _ in range(embeedingSize)] for _ in range(sequenceSize)]
    for i in range(sequenceSize):
        for j in range(embeedingSize):
            fnnResnetOutput[i][j] = h2Output[i][j]+layerNormOutput[i][j]
    
    fnnLayerNormOutput = [[0.0 for _ in range(embeedingSize)] for _ in range(sequenceSize)]
    for i in range(sequenceSize):
        mean = sum(fnnResnetOutput[i])/embeedingSize
        variance = sum([(x-mean)**2 for x in fnnResnetOutput[i]])/embeedingSize
        for j in range(embeedingSize):
            fnnLayerNormOutput[i][j] = (fnnResnetOutput[i][j]-mean)/math.sqrt(variance+epsilon)
    
    
    #next word prediction
    for i in range(len(tokens)-1):
        logits = [0.0]*wordsCount
        target = tokens[i+1]
        total_loss = 0
        for j in range(wordsCount):
            for k in range(embeedingSize):
                logits[j]+=Wy[k][j]*fnnLayerNormOutput[i][k]
        expLogits = [math.exp(val) for val in logits]
        total = sum(expLogits)
        probabilities = [val/total for val in expLogits]
        loss = -math.log(probabilities[target-1])
        total_loss+=loss

        #BACK PROPOGATION
        y_actual = [0]*wordsCount
        y_actual[target-1] = 1
        error = [probabilities[ind]-y_actual[ind] for ind in range(wordsCount)]

        DXy = [0.0]*embeedingSize

        for k in range(embeedingSize):
            for j in range(wordsCount):
                DXy[k]+=(error[j]*Wy[k][j])

        for row in range(embeedingSize):
            for col in range(wordsCount):
                Wy[row][col]-=lr*error[col]*fnnLayerNormOutput[i][row]

        dH1 = [0.0] * hlayer1Dimensions

        for k in range(hlayer1Dimensions):
            for j in range(embeedingSize):
                dH1[k] += (
                    DXy[j]
                    *
                    Wh2[k][j]
                )
        
        for k in range(hlayer1Dimensions):
            for j in range(embeedingSize):
                Wh2[k][j]-=(lr*DXy[j]*h1Output[i][k])


        for k in range(hlayer1Dimensions):
            if h1Output[i][k]<=0:
                dH1[k]=0

        dLayerNorm = [0.0] * embeedingSize

        for k in range(embeedingSize):

            for j in range(hlayer1Dimensions):

                dLayerNorm[k] += (
                    dH1[j]
                    *
                    Wh1[k][j]
                )

        for k in range(embeedingSize):

            for j in range(hlayer1Dimensions):

                Wh1[k][j] -= (
                    lr
                    *
                    dH1[j]
                    *
                    layerNormOutput[i][k]
                )
        
        dProjected = dLayerNorm[:]
        dEmbeedings = dLayerNorm[:]

        dConcatinated = [0.0]*(heads*attentionDimensions)

        for k in range(heads * attentionDimensions):
            for j in range(embeedingSize):

                dConcatinated[k] += (
                    Wo[k][j]
                    *
                    dProjected[j]
                )

        for k in range(heads*attentionDimensions):
            for j in range(embeedingSize):
                Wo[k][j]-=(lr*concatinatedValue[i][k]*dProjected[j])

        dHeadValues = []
        for k in range(heads):
            start = head*attentionDimensions
            end = start+attentionDimensions
            dHeadVertex = dConcatinated[start:end]
            dHeadValues.append(dHeadVertex)
        dV = [
            [0.0 for _ in range(attentionDimensions)]
            for _ in range(sequenceSize)
        ]

        for j in range(sequenceSize):

            for k in range(attentionDimensions):

                dV[j][k] += (
                    attentionWeights[i][j]
                    *
                    dHeadValues[head][k]
                )

        dAttentionWeights = [0.0] * sequenceSize

        for j in range(sequenceSize):

            for k in range(attentionDimensions):

                dAttentionWeights[j] += (
                    dHeadValues[head][k]
                    *
                    V[j][k]
                )

        dScores = [0.0] * sequenceSize

        # compute weighted sum
        weightedSum = 0.0

        for j in range(sequenceSize):

            weightedSum += (
                dAttentionWeights[j]
                *
                attentionWeights[i][j]
            )

        # compute softmax gradients
        for j in range(sequenceSize):

            dScores[j] = (
                attentionWeights[i][j]
                *
                (
                    dAttentionWeights[j]
                    -
                    weightedSum
                )
            )
        dQ = [
            [0.0 for _ in range(attentionDimensions)]
            for _ in range(sequenceSize)
        ]

        dK = [
            [0.0 for _ in range(attentionDimensions)]
            for _ in range(sequenceSize)
        ]

        for j in range(sequenceSize):

            for k in range(attentionDimensions):

                dQ[i][k] += (
                    dScores[j]
                    *
                    K[j][k]
                    /
                    math.sqrt(attentionDimensions)
                )

                dK[j][k] += (
                    dScores[j]
                    *
                    Q[i][k]
                    /
                    math.sqrt(attentionDimensions)
                )
        
        dX_total = [[0.0 for _ in range(embeedingSize)] for _ in range(sequenceSize)]

        for head in range(heads):

            # reset per-head gradients
            dX = [[0.0 for _ in range(embeedingSize)] for _ in range(sequenceSize)]

            # 1. BACKPROP Wq, Wk, Wv
            for tokenPos in range(sequenceSize):

                x = embeedingMatrix[tokens[tokenPos] - 1]

                for inDim in range(embeedingSize):
                    for outDim in range(attentionDimensions):

                        Wq[head][inDim][outDim] -= lr * x[inDim] * dQ[head][tokenPos][outDim]
                        Wk[head][inDim][outDim] -= lr * x[inDim] * dK[head][tokenPos][outDim]
                        Wv[head][inDim][outDim] -= lr * x[inDim] * dV[head][tokenPos][outDim]

                        # gradient to embeddings
                        dX[tokenPos][inDim] += (
                            dQ[head][tokenPos][outDim] * Wq[head][inDim][outDim]
                            + dK[head][tokenPos][outDim] * Wk[head][inDim][outDim]
                            + dV[head][tokenPos][outDim] * Wv[head][inDim][outDim]
                        )

            # accumulate across heads
            for i in range(sequenceSize):
                for j in range(embeedingSize):
                    dX_total[i][j] += dX[i][j]

        # 2. UPDATE EMBEDDINGS

        for tokenPos in range(sequenceSize):

            wordId = tokens[tokenPos] - 1

            for dim in range(embeedingSize):

                embeedingMatrix[wordId][dim] -= lr * dX[tokenPos][dim]


        


values:  [[0.10294582557224789, 0.01154258274508273, 0.30744288459786157, -0.19709005090384368, -0.0789902821268277], [0.0903297317849903, 0.011275548764957746, 0.27672735826292083, -0.18096500122119247, -0.06800388881315891], [0.10164970998102378, 0.007386299917671818, 0.2811655503523366, -0.17909230863167697, -0.06986722190891258], [0.10871260859593887, 0.011823882929366678, 0.29701552795597863, -0.18664326447546656, -0.07587386570782242]]
